# Informe Técnico - Preprocesamiento de Datos (EFT)
## Caso de Estudio: Banco Financiero Global - Campaña de Depósitos a Plazo

**Asignatura:** ADY1100 - Preprocesamiento de Datos  
**Integrantes:**
- Vicente Benjamín Alarcón Gallardo
- Carlos Ignacio Bittner Navea
- Francisco Jesús Aránguiz Inostroza

---

### Contexto del Negocio

El **Banco Financiero Global**, una institución líder en servicios financieros, enfrenta una tasa de conversión inferior a las expectativas en la captación de **depósitos a plazo** - uno de sus productos estratégicos principales. A pesar de inversiones significativas en campañas de telemarketing, el rendimiento limitado afecta los objetivos de captación y fidelización de clientes.

La alta gerencia ha decidido impulsar el uso de **herramientas analíticas avanzadas** para desarrollar un modelo predictivo que identifique a los clientes con mayor probabilidad de contratar un depósito a plazo. Antes de construir dicho modelo, es **imprescindible preparar los datos** para asegurar su calidad y usabilidad.

### Objetivo de este Informe

Este notebook realiza el **preprocesamiento completo** del dataset `bank-additional-full.csv`, que contiene 41,188 registros de campañas de marketing realizadas entre mayo de 2008 y noviembre de 2010. El análisis abarca:

1. Exploración y comprensión de la estructura de datos
2. Tratamiento de valores faltantes (incluyendo imputación con KNN)
3. Análisis exploratorio univariado, bivariado y multivariado (15+ gráficos)
4. Detección y tratamiento de outliers
5. Codificación de variables categóricas
6. Escalamiento de variables numéricas
7. Consideraciones éticas en el manejo de datos

---
## 1. Configuración del Entorno y Librerías

In [ ]:
# Importación de librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Suprimir advertencias innecesarias para mantener el notebook limpio
warnings.filterwarnings('ignore')

# Configuración global de estilo para visualizaciones premium
sns.set_theme(style="whitegrid", font_scale=1.1)
sns.set_palette("husl")

# Configuración de matplotlib para gráficos de alta calidad
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.labelsize'] = 12

print("Entorno configurado correctamente.")

---
## 2. Carga y Exploración Inicial del Dataset

El dataset `bank-additional-full.csv` contiene información de campañas de telemarketing del banco. El archivo utiliza punto y coma (`;`) como separador, por lo que debemos especificarlo al cargarlo.

In [ ]:
# Cargar el dataset completo
df = pd.read_csv('bank-additional-full.csv', sep=';')

# Verificar las dimensiones del dataset
print(f"Dimensiones del dataset: {df.shape[0]:,} filas x {df.shape[1]} columnas")
print(f"Memoria utilizada: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
# Previsualización de las primeras filas
df.head(10)

In [ ]:
# Información general del dataset: tipos de datos y valores no nulos
df.info()

In [ ]:
# Estadísticas descriptivas de las variables numéricas
df.describe().round(2)

In [ ]:
# Estadísticas descriptivas de las variables categóricas
df.describe(include='object')

### Clasificación de Variables

Identificamos los tipos de variables presentes en el dataset para planificar el análisis adecuado:

In [ ]:
# Separar variables por tipo
var_numericas = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
var_categoricas = df.select_dtypes(include=['object']).columns.tolist()

print("Variables Numéricas:")
for v in var_numericas:
    print(f"  - {v}: rango [{df[v].min()}, {df[v].max()}], valores únicos: {df[v].nunique()}")

print(f"\n Variables Categóricas:")
for v in var_categoricas:
    print(f"  - {v}: {df[v].nunique()} categorías -> {df[v].unique().tolist()}")

---
## 3. Análisis de Valores Faltantes y Tratamiento con KNN

En este dataset, los valores faltantes no aparecen como `NaN` sino codificados como la etiqueta `"unknown"` en las variables categóricas. Es fundamental identificarlos, cuantificarlos y decidir cómo tratarlos.

### 3.1 Identificación y Cuantificación de `unknown`

In [ ]:
# Identificar columnas con valores "unknown" y contar su frecuencia
unknown_counts = {}
for col in var_categoricas:
    count = (df[col] == 'unknown').sum()
    if count > 0:
        unknown_counts[col] = count

# Crear DataFrame resumen
df_unknowns = pd.DataFrame({
    'Variable': unknown_counts.keys(),
    'Cantidad de unknown': unknown_counts.values(),
    'Porcentaje (%)': [round(v / len(df) * 100, 2) for v in unknown_counts.values()]
}).sort_values('Cantidad de unknown', ascending=False).reset_index(drop=True)

print("Resumen de valores 'unknown' en el dataset:")
print(df_unknowns.to_string(index=False))

### Gráfico 1: Distribución de Valores `unknown` por Variable

Este gráfico permite al equipo de Ciencia de Datos del banco **identificar rápidamente qué variables presentan mayor cantidad de información ausente**, para priorizar las estrategias de limpieza y decidir qué columnas requieren un tratamiento más cuidadoso antes de alimentar un modelo predictivo.

In [ ]:
# GRÁFICO 1: Barras horizontales de valores unknown por variable
fig, ax = plt.subplots(figsize=(10, 5))
colors = sns.color_palette("muted", len(df_unknowns))
bars = ax.barh(df_unknowns['Variable'], df_unknowns['Cantidad de unknown'], color=colors, edgecolor='white')

# Añadir etiquetas de porcentaje
for bar, pct in zip(bars, df_unknowns['Porcentaje (%)']):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
            f'{pct}%', va='center', fontweight='bold', fontsize=11)

ax.set_xlabel('Cantidad de registros con valor "unknown"')
ax.set_ylabel('Variable')
ax.set_title('Distribución de Valores Faltantes ("unknown") por Variable')
plt.tight_layout()
plt.show()

### 3.2 Estrategia de Tratamiento

Dado que las variables con `"unknown"` son **categóricas**, la estrategia inicial más conservadora es **reemplazarlos por la moda** (valor más frecuente) de cada columna. Sin embargo, para validar que esta imputación no introduce sesgos importantes, utilizaremos **KNN Imputer** como método alternativo y compararemos los resultados.

#### ¿Por qué KNN Imputer?
El algoritmo KNN (K-Nearest Neighbors) imputa valores faltantes basándose en la **similitud con otros registros**. Esto respeta la estructura interna de los datos mejor que reemplazar ciegamente con la moda, ya que considera el contexto de las demás variables del cliente.

### 3.3 Imputación con la Moda vs. KNN

In [ ]:
from sklearn.impute import KNNImputer
from sklearn.preprocessing import LabelEncoder

# Crear copia del DataFrame para trabajar
df_work = df.copy()

# --- Método 1: Imputación por la Moda ---
df_moda = df.copy()
for col in unknown_counts.keys():
    # La moda se calcula excluyendo 'unknown'
    moda_val = df_moda[df_moda[col] != 'unknown'][col].mode()[0]
    df_moda[col] = df_moda[col].replace('unknown', moda_val)
    print(f" [Moda] {col}: 'unknown' -> '{moda_val}' ({unknown_counts[col]:,} registros)")

print(f"\n Imputación por moda completada.")

In [ ]:
# --- Método 2: Imputación con KNN ---
# Para usar KNN, necesitamos convertir las categóricas a numéricas temporalmente
df_knn = df.copy()

# Paso 1: Marcar los unknowns como NaN para que KNN los identifique
for col in unknown_counts.keys():
    df_knn[col] = df_knn[col].replace('unknown', np.nan)

# Paso 2: Codificar las categóricas con LabelEncoder (temporal)
label_encoders = {}
for col in var_categoricas:
    le = LabelEncoder()
    # Ajustar solo con valores no nulos
    mask = df_knn[col].notna()
    le.fit(df_knn.loc[mask, col])
    df_knn.loc[mask, col] = le.transform(df_knn.loc[mask, col])
    df_knn.loc[~mask, col] = np.nan  # Mantener NaN para KNN
    label_encoders[col] = le

# Convertir todo a numérico
df_knn_numeric = df_knn.apply(pd.to_numeric, errors='coerce')

# Paso 3: Aplicar KNNImputer (k=5 vecinos)
knn_imputer = KNNImputer(n_neighbors=5, weights='distance')
df_knn_imputed = pd.DataFrame(
    knn_imputer.fit_transform(df_knn_numeric),
    columns=df_knn_numeric.columns
)

# Paso 4: Decodificar las categóricas de vuelta a sus etiquetas originales
for col in var_categoricas:
    le = label_encoders[col]
    # Redondear al entero más cercano (KNN produce flotantes)
    df_knn_imputed[col] = df_knn_imputed[col].round().astype(int)
    # Asegurar que los valores estén dentro del rango válido
    max_val = len(le.classes_) - 1
    df_knn_imputed[col] = df_knn_imputed[col].clip(0, max_val)
    df_knn_imputed[col] = le.inverse_transform(df_knn_imputed[col])

print("Imputación con KNN (k=5) completada.")

### 3.4 Comparación: Moda vs. KNN

A continuación comparamos las distribuciones resultantes de ambos métodos para verificar si la imputación con la moda introduce algún sesgo significativo que estemos pasando por alto.

In [ ]:
# Comparación visual de las distribuciones post-imputación
cols_with_unknown = list(unknown_counts.keys())
n_cols = len(cols_with_unknown)

fig, axes = plt.subplots(n_cols, 2, figsize=(16, 4 * n_cols))
if n_cols == 1:
    axes = [axes]

for i, col in enumerate(cols_with_unknown):
    # Distribución con Moda
    df_moda[col].value_counts().plot(kind='bar', ax=axes[i][0],
                                      color=sns.color_palette("muted"), edgecolor='white')
    axes[i][0].set_title(f'{col} - Imputación por Moda', fontweight='bold')
    axes[i][0].set_ylabel('Frecuencia')
    axes[i][0].tick_params(axis='x', rotation=45)

    # Distribución con KNN
    df_knn_imputed[col].value_counts().plot(kind='bar', ax=axes[i][1],
                                             color=sns.color_palette("Set2"), edgecolor='white')
    axes[i][1].set_title(f'{col} - Imputación con KNN (k=5)', fontweight='bold')
    axes[i][1].set_ylabel('Frecuencia')
    axes[i][1].tick_params(axis='x', rotation=45)

plt.suptitle('Comparación de Distribuciones: Moda vs. KNN Imputer', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 3.5 Análisis de la Comparación

Al comparar las distribuciones resultantes de ambos métodos, podemos observar si la imputación por moda **concentra excesivamente** los valores en una sola categoría (la más frecuente), mientras que KNN distribuye los valores imputados de forma más proporcional al contexto de cada registro.

**Decisión:** Para el resto del análisis, utilizaremos el DataFrame imputado con **KNN**, ya que preserva mejor la estructura natural de los datos. Sin embargo, es importante documentar que la diferencia entre ambos métodos es marginal para las variables con bajo porcentaje de `unknown`, lo que valida que la moda también sería una opción aceptable en esos casos.

In [ ]:
# Seleccionar el DataFrame imputado con KNN para el resto del análisis
# Mantener los tipos de datos numéricos originales
for col in var_numericas:
    df_knn_imputed[col] = pd.to_numeric(df_knn_imputed[col], errors='coerce')

df_clean = df_knn_imputed.copy()

# Verificar que no quedan unknowns ni NaN
print("Verificación post-imputación:")
print(f"  Valores NaN restantes: {df_clean.isnull().sum().sum()}")
for col in cols_with_unknown:
    unknown_remaining = (df_clean[col] == 'unknown').sum()
    print(f"  '{col}' con 'unknown': {unknown_remaining}")
print("\n Dataset limpio y listo para el análisis exploratorio.")

---
## 4. Análisis Exploratorio Univariado - Variables Categóricas

El análisis univariado permite comprender la **distribución individual** de cada variable. Para las variables categóricas, utilizamos gráficos de barras (`countplot`) que muestran la frecuencia de cada categoría, permitiendo al banco identificar rápidamente los segmentos predominantes en su base de clientes.

### Gráfico 2: Distribución de la Variable Objetivo (`y`)

**Justificación de negocio:** Este es el gráfico más importante del análisis, ya que muestra el **balance de clases** de la variable que queremos predecir. Un desbalance significativo entre los clientes que suscribieron (`yes`) y los que no (`no`) tiene implicancias directas en la selección del algoritmo de machine learning y en las métricas de evaluación que el banco debería utilizar.

In [ ]:
# GRÁFICO 2: Distribución de la variable objetivo 'y'
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Countplot
palette_y = {'no': '#E74C3C', 'yes': '#2ECC71'}
sns.countplot(data=df_clean, x='y', palette=palette_y, edgecolor='white', ax=axes[0])
axes[0].set_title('Distribución de Suscripción a Depósito a Plazo')
axes[0].set_xlabel('¿Suscribió depósito a plazo?')
axes[0].set_ylabel('Cantidad de clientes')

# Añadir etiquetas de valor
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}',
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontweight='bold', fontsize=12)

# Gráfico de torta (proporción)
counts = df_clean['y'].value_counts()
axes[1].pie(counts, labels=['No suscribió', 'Sí suscribió'],
            autopct='%1.1f%%', colors=['#E74C3C', '#2ECC71'],
            startangle=90, explode=(0, 0.05),
            textprops={'fontsize': 12, 'fontweight': 'bold'})
axes[1].set_title('Proporción de Suscripción')

plt.suptitle('Variable Objetivo: Suscripción a Depósito a Plazo', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

# Imprimir el desbalance
ratio = counts['no'] / counts['yes']
print(f"Ratio de desbalance (no/yes): {ratio:.1f}:1")

### Gráfico 3: Distribución por Tipo de Trabajo (`job`)

**Justificación de negocio:** Conocer la composición laboral de la base de clientes permite al Banco Financiero Global **personalizar sus estrategias de contacto**. Por ejemplo, si la mayoría de los clientes son técnicos o administradores, el banco puede adaptar sus horarios de llamada y el lenguaje comercial para maximizar la efectividad.

In [ ]:
# GRÁFICO 3: Distribución de tipos de trabajo
fig, ax = plt.subplots(figsize=(14, 6))
order = df_clean['job'].value_counts().index
sns.countplot(data=df_clean, y='job', order=order, palette='husl', edgecolor='white', ax=ax)
ax.set_title('Distribución de Clientes por Tipo de Trabajo')
ax.set_xlabel('Cantidad de clientes')
ax.set_ylabel('Tipo de trabajo')

# Añadir etiquetas
for p in ax.patches:
    ax.annotate(f'{int(p.get_width()):,}',
                (p.get_width(), p.get_y() + p.get_height() / 2.),
                ha='left', va='center', fontweight='bold', fontsize=10, xytext=(5, 0),
                textcoords='offset points')
plt.tight_layout()
plt.show()

### Gráfico 4: Distribución por Estado Civil (`marital`)

**Justificación de negocio:** El estado civil es un indicador demográfico que influye en las decisiones financieras. Los clientes casados pueden tener mayor interés en productos de ahorro a largo plazo para su familia, mientras que los solteros podrían priorizar liquidez. Esta información ayuda al banco a **segmentar campañas** por perfil familiar.

In [ ]:
# GRÁFICO 4: Distribución de estado civil
fig, ax = plt.subplots(figsize=(10, 5))
order = df_clean['marital'].value_counts().index
sns.countplot(data=df_clean, x='marital', order=order, palette='Set2', edgecolor='white', ax=ax)
ax.set_title('Distribución de Clientes por Estado Civil')
ax.set_xlabel('Estado civil')
ax.set_ylabel('Cantidad de clientes')

for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontweight='bold', fontsize=11)
plt.tight_layout()
plt.show()

### Gráfico 5: Distribución por Nivel Educativo (`education`)

**Justificación de negocio:** El nivel de educación está correlacionado con el nivel de ingresos y la comprensión de productos financieros. Identificar qué segmentos educativos predominan permite al banco **adaptar la complejidad de sus materiales de marketing** y decidir si necesita campañas educativas complementarias para ciertos grupos.

In [ ]:
# GRÁFICO 5: Distribución de nivel educativo
fig, ax = plt.subplots(figsize=(14, 6))
order = df_clean['education'].value_counts().index
sns.countplot(data=df_clean, y='education', order=order, palette='muted', edgecolor='white', ax=ax)
ax.set_title('Distribución de Clientes por Nivel Educativo')
ax.set_xlabel('Cantidad de clientes')
ax.set_ylabel('Nivel educativo')

for p in ax.patches:
    ax.annotate(f'{int(p.get_width()):,}',
                (p.get_width(), p.get_y() + p.get_height() / 2.),
                ha='left', va='center', fontweight='bold', fontsize=10, xytext=(5, 0),
                textcoords='offset points')
plt.tight_layout()
plt.show()

### Gráfico 6: Distribución por Tipo de Contacto (`contact`)

**Justificación de negocio:** Entender qué canal de comunicación prevalece (celular vs. teléfono fijo) es crucial para que el banco **optimice la asignación de recursos** en sus centrales de llamadas y evalúe si vale la pena invertir en canales digitales complementarios (SMS, WhatsApp, email) en futuras campañas.

In [ ]:
# GRÁFICO 6: Distribución de tipo de contacto
fig, ax = plt.subplots(figsize=(8, 5))
sns.countplot(data=df_clean, x='contact', palette=['#3498DB', '#E67E22'], edgecolor='white', ax=ax)
ax.set_title('Distribución por Tipo de Contacto')
ax.set_xlabel('Tipo de comunicación')
ax.set_ylabel('Cantidad de clientes')

for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

### Gráfico 7: Distribución del Resultado de Campañas Anteriores (`poutcome`)

**Justificación de negocio:** El historial de éxito o fracaso de campañas previas es un indicador potente de comportamiento futuro. Si un cliente tuvo una experiencia positiva en una campaña anterior, es más probable que vuelva a suscribir. Esta variable permite al banco **priorizar clientes con historial favorable** y evitar el desgaste de contactar repetidamente a quienes ya rechazaron la oferta.

In [ ]:
# GRÁFICO 7: Resultado de campañas anteriores
fig, ax = plt.subplots(figsize=(10, 5))
order = df_clean['poutcome'].value_counts().index
palette_pout = {'nonexistent': '#95A5A6', 'failure': '#E74C3C', 'success': '#2ECC71'}
sns.countplot(data=df_clean, x='poutcome', order=order, palette=palette_pout, edgecolor='white', ax=ax)
ax.set_title('Resultado de la Campaña de Marketing Anterior')
ax.set_xlabel('Resultado de la campaña anterior')
ax.set_ylabel('Cantidad de clientes')

for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

---
## 5. Análisis Exploratorio Univariado - Variables Numéricas

Para las variables numéricas, utilizamos histogramas con estimación de densidad kernel (KDE) y boxplots. Estas visualizaciones permiten identificar la **forma de la distribución**, su **tendencia central**, **dispersión** y la presencia de **valores atípicos (outliers)**.

### Gráfico 8: Distribución de Edad (`age`)

**Justificación de negocio:** La edad del cliente es un factor demográfico esencial para la segmentación de mercado. Entender el rango de edades predominante en la base de datos permite al banco **diseñar productos financieros diferenciados** (ej. depósitos con beneficios para jóvenes o condiciones especiales para jubilados).

In [ ]:
# GRÁFICO 8: Histograma + KDE de la edad
fig, ax = plt.subplots(figsize=(12, 5))
sns.histplot(data=df_clean, x='age', bins=40, kde=True, color='#3498DB',
             edgecolor='white', alpha=0.7, ax=ax)
ax.axvline(df_clean['age'].mean(), color='#E74C3C', linestyle='--', linewidth=2,
           label=f'Media: {df_clean["age"].mean():.1f} años')
ax.axvline(df_clean['age'].median(), color='#2ECC71', linestyle='--', linewidth=2,
           label=f'Mediana: {df_clean["age"].median():.1f} años')
ax.set_title('Distribución de Edad de los Clientes')
ax.set_xlabel('Edad (años)')
ax.set_ylabel('Frecuencia')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

### Gráfico 9: Distribución de Duración de Llamada (`duration`)

**Justificación de negocio:** La duración de la última llamada es un indicador de **engagement** del cliente. Sin embargo, es crucial señalar que esta variable presenta un problema ético significativo: **no se conoce antes de realizar la llamada** y, por lo tanto, no puede usarse para predecir si vale la pena llamar a un cliente. Su inclusión en un modelo predictivo constituiría *data leakage*.

>  **Nota ética:** Esta variable se incluye aquí únicamente con fines exploratorios. Será descartada antes de cualquier modelado predictivo, tal como recomienda la documentación original del dataset (Moro et al., 2014).

In [ ]:
# GRÁFICO 9: Histograma + KDE de la duración de llamada
fig, ax = plt.subplots(figsize=(12, 5))
sns.histplot(data=df_clean, x='duration', bins=50, kde=True, color='#E67E22',
             edgecolor='white', alpha=0.7, ax=ax)
ax.axvline(df_clean['duration'].mean(), color='#E74C3C', linestyle='--', linewidth=2,
           label=f'Media: {df_clean["duration"].mean():.0f} seg')
ax.axvline(df_clean['duration'].median(), color='#2ECC71', linestyle='--', linewidth=2,
           label=f'Mediana: {df_clean["duration"].median():.0f} seg')
ax.set_title('Distribución de Duración de la Última Llamada (en segundos)')
ax.set_xlabel('Duración (segundos)')
ax.set_ylabel('Frecuencia')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f" Clientes con duración = 0: {(df_clean['duration'] == 0).sum()}")
print(f"  (Todos estos clientes tienen y = 'no', confirmando el data leakage)")

### Gráfico 10: Boxplots de Variables Numéricas Clave

**Justificación de negocio:** Los boxplots permiten detectar visualmente los **valores atípicos (outliers)** que podrían sesgar los análisis y modelos del banco. Identificarlos tempranamente permite al equipo de datos tomar decisiones informadas sobre su tratamiento: ¿son errores de registro o clientes con comportamientos extremos pero legítimos?

In [ ]:
# GRÁFICO 10: Boxplots de variables numéricas seleccionadas
cols_boxplot = ['age', 'campaign', 'previous', 'emp.var.rate',
                'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()

for i, col in enumerate(cols_boxplot):
    sns.boxplot(data=df_clean, y=col, color=sns.color_palette("husl", len(cols_boxplot))[i],
                width=0.5, ax=axes[i])
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_ylabel('')

plt.suptitle('Boxplots de Variables Numéricas - Detección de Outliers', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6. Análisis Bivariado y Multivariado

El análisis bivariado permite explorar la **relación entre dos variables**, especialmente entre cada variable de entrada y la variable objetivo `y`. Esto es fundamental para que el banco identifique los **factores más influyentes** en la conversión de depósitos a plazo.

### Gráfico 11: Tasa de Suscripción por Tipo de Trabajo

**Justificación de negocio:** Este gráfico revela qué perfiles laborales presentan **mayor propensión** a contratar un depósito a plazo. Si los estudiantes o jubilados muestran tasas de conversión más altas, el banco puede **reasignar presupuesto de marketing** hacia esos segmentos, maximizando el retorno sobre la inversión publicitaria.

In [ ]:
# GRÁFICO 11: Tasa de suscripción por tipo de trabajo
# Calcular la tasa de suscripción por trabajo
df_clean['y_numeric'] = (df_clean['y'] == 'yes').astype(int)
tasa_job = df_clean.groupby('job')['y_numeric'].mean().sort_values(ascending=False) * 100

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.barh(tasa_job.index, tasa_job.values, color=sns.color_palette("husl", len(tasa_job)),
               edgecolor='white')
ax.set_title('Tasa de Suscripción a Depósito a Plazo por Tipo de Trabajo')
ax.set_xlabel('Tasa de suscripción (%)')
ax.set_ylabel('Tipo de trabajo')

for bar, val in zip(bars, tasa_job.values):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontweight='bold', fontsize=10)
plt.tight_layout()
plt.show()

### Gráfico 12: Tasa de Suscripción por Nivel Educativo

**Justificación de negocio:** Relacionar la educación con la tasa de conversión permite al banco determinar si los clientes con mayor nivel educativo son más receptivos a los depósitos a plazo. Esta información orienta la **estrategia de comunicación**: clientes con título universitario podrían responder mejor a argumentos técnicos sobre rendimiento financiero, mientras que otros segmentos podrían necesitar explicaciones más accesibles.

In [ ]:
# GRÁFICO 12: Tasa de suscripción por nivel educativo
tasa_edu = df_clean.groupby('education')['y_numeric'].mean().sort_values(ascending=False) * 100

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(tasa_edu.index, tasa_edu.values, color=sns.color_palette("muted", len(tasa_edu)),
               edgecolor='white')
ax.set_title('Tasa de Suscripción a Depósito a Plazo por Nivel Educativo')
ax.set_xlabel('Tasa de suscripción (%)')
ax.set_ylabel('Nivel educativo')

for bar, val in zip(bars, tasa_edu.values):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontweight='bold', fontsize=10)
plt.tight_layout()
plt.show()

### Gráfico 13: Tasa de Suscripción por Mes del Año (Estacionalidad)

**Justificación de negocio:** Analizar la estacionalidad de las conversiones permite al banco **planificar el calendario de campañas** de forma estratégica. Si ciertos meses (como marzo o diciembre) presentan tasas de conversión significativamente más altas, el banco debería concentrar sus esfuerzos de telemarketing en esos períodos para maximizar resultados.

In [ ]:
# GRÁFICO 13: Tasa de suscripción por mes
month_order = ['jan', 'feb', 'mar', 'apr', 'may', 'jun',
               'jul', 'aug', 'sep', 'oct', 'nov', 'dec']
# Filtrar solo los meses presentes en el dataset
month_order = [m for m in month_order if m in df_clean['month'].unique()]

tasa_month = df_clean.groupby('month')['y_numeric'].mean().reindex(month_order) * 100

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(tasa_month.index, tasa_month.values, color=sns.color_palette("husl", len(tasa_month)),
              edgecolor='white')
ax.set_title('Tasa de Suscripción por Mes del Año - Análisis de Estacionalidad')
ax.set_xlabel('Mes del último contacto')
ax.set_ylabel('Tasa de suscripción (%)')

for bar, val in zip(bars, tasa_month.values):
    if not np.isnan(val):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=10)

ax.axhline(y=df_clean['y_numeric'].mean() * 100, color='red', linestyle='--',
           label=f'Promedio general: {df_clean["y_numeric"].mean()*100:.1f}%')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

### Gráfico 14: Distribución de Edad según Suscripción

**Justificación de negocio:** Este boxplot segmentado permite identificar si existe un **rango etario** que muestre mayor receptividad a los depósitos a plazo. Si los clientes mayores de 60 años tienen una mediana de suscripción más alta, el banco puede crear productos con condiciones especiales para ese grupo demográfico.

In [ ]:
# GRÁFICO 14: Boxplot de edad segmentado por variable objetivo
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=df_clean, x='y', y='age', palette={'no': '#E74C3C', 'yes': '#2ECC71'},
            width=0.5, ax=ax)
ax.set_title('Distribución de Edad según Suscripción a Depósito a Plazo')
ax.set_xlabel('¿Suscribió depósito a plazo?')
ax.set_ylabel('Edad (años)')

# Añadir medianas como texto
medians = df_clean.groupby('y')['age'].median()
for i, (label, median) in enumerate(medians.items()):
    ax.text(i, median + 1, f'Mediana: {median:.0f}', ha='center', fontweight='bold', fontsize=11)
plt.tight_layout()
plt.show()

### Gráfico 15: Distribución del Euribor 3M según Suscripción

**Justificación de negocio:** El Euribor a 3 meses es un indicador macroeconómico clave que refleja el costo del dinero en Europa. Su relación con la variable objetivo revela cómo el **contexto económico general** afecta las decisiones de los clientes. Si las suscripciones aumentan cuando el Euribor es bajo, el banco puede activar campañas más agresivas en períodos de tipos de interés reducidos.

In [ ]:
# GRÁFICO 15: Boxplot de euribor3m segmentado por variable objetivo
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=df_clean, x='y', y='euribor3m', palette={'no': '#E74C3C', 'yes': '#2ECC71'},
            width=0.5, ax=ax)
ax.set_title('Tasa Euribor 3M según Suscripción a Depósito a Plazo')
ax.set_xlabel('¿Suscribió depósito a plazo?')
ax.set_ylabel('Euribor 3 meses (%)')

medians = df_clean.groupby('y')['euribor3m'].median()
for i, (label, median) in enumerate(medians.items()):
    ax.text(i, median + 0.1, f'Mediana: {median:.2f}', ha='center', fontweight='bold', fontsize=11)
plt.tight_layout()
plt.show()

### Gráfico 16: Mapa de Calor de Correlación entre Variables Numéricas

**Justificación de negocio:** La matriz de correlación permite identificar **relaciones lineales** entre variables numéricas. Correlaciones altas entre variables de entrada (multicolinealidad) pueden causar problemas en modelos predictivos. Además, variables con alta correlación con la variable objetivo son candidatas principales como *features* predictivas para el modelo del banco.

In [ ]:
# GRÁFICO 16: Heatmap de correlación
fig, ax = plt.subplots(figsize=(14, 10))
corr_matrix = df_clean[var_numericas + ['y_numeric']].corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5,
            square=True, ax=ax,
            cbar_kws={'shrink': 0.8, 'label': 'Coeficiente de correlación'})
ax.set_title('Matriz de Correlación - Variables Numéricas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Identificar las correlaciones más fuertes con la variable objetivo
print("Correlaciones más fuertes con la variable objetivo (y):")
corr_with_y = corr_matrix['y_numeric'].drop('y_numeric').abs().sort_values(ascending=False)
for var, corr in corr_with_y.head(5).items():
    direction = "positiva" if corr_matrix.loc[var, 'y_numeric'] > 0 else "negativa"
    print(f"  - {var}: {corr:.3f} ({direction})")

### Gráfico 17: Pairplot de Variables Socioeconómicas

**Justificación de negocio:** Este gráfico multivariado permite visualizar simultáneamente las relaciones entre los **indicadores macroeconómicos** y su comportamiento diferenciado entre clientes que suscribieron y los que no. Es fundamental para que el banco comprenda cómo el entorno económico nacional impacta en las decisiones financieras individuales de sus clientes.

In [ ]:
# GRÁFICO 17: Pairplot de variables socioeconómicas coloreado por y
socio_vars = ['emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'y']
df_pairplot = df_clean[socio_vars].sample(n=3000, random_state=42)  # Muestra para rendimiento

g = sns.pairplot(df_pairplot, hue='y', palette={'no': '#E74C3C', 'yes': '#2ECC71'},
                 diag_kind='kde', plot_kws={'alpha': 0.4, 's': 15},
                 diag_kws={'fill': True, 'alpha': 0.5})
g.figure.suptitle('Relaciones entre Indicadores Socioeconómicos por Suscripción',
                   fontsize=14, fontweight='bold', y=1.02)
plt.show()

---
## 7. Detección y Tratamiento de Outliers (Método IQR)

Los valores atípicos (outliers) pueden distorsionar los análisis estadísticos y el rendimiento de los modelos de machine learning. Utilizamos el **método del Rango Intercuartílico (IQR)** para identificarlos de forma objetiva:

- **Q1** = Primer cuartil (percentil 25)
- **Q3** = Tercer cuartil (percentil 75)
- **IQR** = Q3 − Q1
- **Outlier** = valor < Q1 − 1.5 x IQR  **o**  valor > Q3 + 1.5 x IQR

In [ ]:
# Detección de outliers con IQR para todas las variables numéricas
print("Detección de Outliers - Método IQR")
print("=" * 65)

outlier_report = []
for col in var_numericas:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df_clean[(df_clean[col] < lower) | (df_clean[col] > upper)]
    pct = len(outliers) / len(df_clean) * 100

    outlier_report.append({
        'Variable': col,
        'Q1': round(Q1, 2),
        'Q3': round(Q3, 2),
        'IQR': round(IQR, 2),
        'Límite inferior': round(lower, 2),
        'Límite superior': round(upper, 2),
        'N° Outliers': len(outliers),
        '% del total': round(pct, 2)
    })
    if len(outliers) > 0:
        print(f" {col:20s} -> {len(outliers):>5,} outliers ({pct:.2f}%) | Rango válido: [{lower:.2f}, {upper:.2f}]")

df_outliers = pd.DataFrame(outlier_report)
print(f"\n Resumen completo disponible en 'df_outliers'")

### Decisión sobre Outliers

No todos los outliers son errores. En el contexto bancario:
- **`age`**: Un cliente de 95 años es atípico pero legítimo. No lo eliminamos.
- **`campaign`**: Un cliente contactado 56 veces es extremo y podría indicar un error operativo o un caso de sobrecontacto. Se recomienda investigar.
- **`duration`**: Los valores extremos en duración son esperables (llamadas muy largas con clientes interesados). Dado que esta variable será descartada del modelo predictivo, no es necesario tratar sus outliers.

**Decisión:** Para este análisis conservamos los outliers, documentando su existencia para que el equipo de modelado pueda tomar decisiones informadas según el algoritmo elegido.

---
## 8. Codificación de Variables Categóricas

Los algoritmos de machine learning requieren datos numéricos. Por lo tanto, debemos transformar las variables categóricas en representaciones numéricas. Aplicaremos dos estrategias según la naturaleza de cada variable:

| Estrategia | Cuándo usarla | Variables |
|---|---|---|
| **Label Encoding (Ordinal)** | Cuando existe un orden jerárquico natural | `education` |
| **One-Hot Encoding (Nominal)** | Cuando las categorías no tienen orden | `job`, `marital`, `contact`, `month`, `day_of_week`, `poutcome`, `default`, `housing`, `loan` |

In [ ]:
# --- Label Encoding para 'education' (variable ordinal) ---
# Definir el orden jerárquico de los niveles educativos
education_order = {
    'illiterate': 0,
    'basic.4y': 1,
    'basic.6y': 2,
    'basic.9y': 3,
    'high.school': 4,
    'professional.course': 5,
    'university.degree': 6
}

df_encoded = df_clean.copy()
df_encoded['education_encoded'] = df_encoded['education'].map(education_order)

# Verificar el mapeo
print("Label Encoding - education:")
print(df_encoded[['education', 'education_encoded']].drop_duplicates().sort_values('education_encoded'))

In [ ]:
# --- One-Hot Encoding para variables nominales ---
nominal_cols = ['job', 'marital', 'contact', 'month', 'day_of_week', 'poutcome',
                'default', 'housing', 'loan']

df_encoded = pd.get_dummies(df_encoded, columns=nominal_cols, drop_first=False, dtype=int)

print(f"Dimensiones después del One-Hot Encoding:")
print(f"  Antes: {df_clean.shape[1]} columnas")
print(f"  Después: {df_encoded.shape[1]} columnas")
print(f"  Nuevas columnas generadas: {df_encoded.shape[1] - df_clean.shape[1]}")

# Mostrar las primeras columnas generadas
new_cols = [c for c in df_encoded.columns if any(c.startswith(n + '_') for n in nominal_cols)]
print(f"\n Ejemplo de nuevas columnas (primeras 10):")
for col in new_cols[:10]:
    print(f"  - {col}")

---
## 9. Escalamiento de Variables Numéricas

Las variables numéricas del dataset tienen rangos muy diferentes (ej. `age` va de 17 a 98, mientras que `nr.employed` va de 4,963 a 5,228). Esta disparidad puede afectar negativamente algoritmos sensibles a la magnitud como KNN, SVM o redes neuronales.

Comparamos dos técnicas de escalamiento:

| Método | Fórmula | Rango resultante | Ideal para |
|---|---|---|---|
| **MinMaxScaler** | (x - min) / (max - min) | [0, 1] | Redes neuronales, datos acotados |
| **StandardScaler** | (x - μ) / σ | Media=0, σ=1 | SVM, regresión logística, PCA |

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Variables a escalar (excluimos 'duration' y la variable objetivo)
cols_to_scale = [c for c in var_numericas if c != 'duration']

# --- StandardScaler ---
scaler_std = StandardScaler()
df_scaled_std = pd.DataFrame(
    scaler_std.fit_transform(df_clean[cols_to_scale]),
    columns=[f'{c}_std' for c in cols_to_scale]
)

# --- MinMaxScaler ---
scaler_mm = MinMaxScaler()
df_scaled_mm = pd.DataFrame(
    scaler_mm.fit_transform(df_clean[cols_to_scale]),
    columns=[f'{c}_mm' for c in cols_to_scale]
)

print("Escalamiento completado.")
print(f"\n StandardScaler - Estadísticas:")
print(df_scaled_std.describe().round(3).loc[['mean', 'std', 'min', 'max']])
print(f"\n MinMaxScaler - Estadísticas:")
print(df_scaled_mm.describe().round(3).loc[['mean', 'std', 'min', 'max']])

### Visualización: Antes vs. Después del Escalamiento

Esta comparación permite verificar visualmente que el escalamiento ha normalizado correctamente los rangos de las variables sin alterar la forma de sus distribuciones.

In [ ]:
# Comparación visual: datos originales vs. escalados
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Datos originales
df_clean[cols_to_scale].boxplot(ax=axes[0], vert=True, patch_artist=True,
                                 boxprops=dict(facecolor='#3498DB', alpha=0.7))
axes[0].set_title('Datos Originales (sin escalar)', fontweight='bold')
axes[0].tick_params(axis='x', rotation=90)

# StandardScaler
df_scaled_std.boxplot(ax=axes[1], vert=True, patch_artist=True,
                       boxprops=dict(facecolor='#2ECC71', alpha=0.7))
axes[1].set_title('StandardScaler (media=0, σ=1)', fontweight='bold')
axes[1].tick_params(axis='x', rotation=90)

# MinMaxScaler
df_scaled_mm.boxplot(ax=axes[2], vert=True, patch_artist=True,
                      boxprops=dict(facecolor='#E67E22', alpha=0.7))
axes[2].set_title('MinMaxScaler (rango [0, 1])', fontweight='bold')
axes[2].tick_params(axis='x', rotation=90)

plt.suptitle('Comparación de Escalamiento de Variables Numéricas', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 10. Consideraciones Éticas en el Manejo de Datos

### 10.1 El Caso de la Variable `duration` - Data Leakage

La variable `duration` (duración de la última llamada en segundos) es el ejemplo más claro de **data leakage** en este dataset:

1. **El problema:** La duración de la llamada **no se conoce antes de realizarla**. Si un modelo predictivo utiliza esta variable para decidir a quién llamar, estaría usando información del futuro (posterior al evento que intenta predecir).
2. **Las consecuencias:** Un modelo con `duration` como feature tendría métricas artificialmente infladas (alta precisión, alto AUC), dando al banco una falsa sensación de confianza.
3. **La decisión ética:** Descartamos `duration` del conjunto de features para el modelo predictivo. Incluirla sería equivalente a hacer trampa: predecir si un cliente comprará algo basándose en cuánto tiempo habló con el vendedor.

### 10.2 Prevención de Sesgos Demográficos

Al analizar variables como `age`, `job` o `education`, es fundamental que el banco evite:
- **Perfiles discriminatorios:** No excluir automáticamente a grupos demográficos con baja tasa de conversión (ej. desempleados), ya que esto podría constituir discriminación financiera.
- **Generalización excesiva:** Las correlaciones observadas son tendencias estadísticas, no determinismos individuales. Cada cliente merece una evaluación justa.

### 10.3 Integridad en la Representación Visual

Todos los gráficos de este informe mantienen la integridad de los datos:
- Los ejes parten de cero (sin truncamiento que exagere diferencias).
- Las escalas son proporcionales y consistentes.
- Se incluyen etiquetas de valor para evitar interpretaciones ambiguas.

---
## 11. Conclusiones y Recomendaciones para el Banco

### Hallazgos Clave del Análisis Exploratorio

1. **Desbalance de clases significativo:** Aproximadamente el 88% de los clientes no suscribieron el depósito. El banco deberá considerar técnicas de balanceo (SMOTE, undersampling) al construir el modelo.

2. **Perfil del cliente con mayor probabilidad de conversión:**
   - Estudiantes y jubilados presentan las tasas de suscripción más altas.
   - Los meses de marzo, septiembre, octubre y diciembre son los más efectivos para campañas.
   - Un Euribor bajo favorece la contratación de depósitos.

3. **Variables socioeconómicas como predictores fuertes:** `euribor3m`, `emp.var.rate` y `nr.employed` muestran las correlaciones más fuertes con la variable objetivo, sugiriendo que el contexto macroeconómico influye tanto o más que las características individuales del cliente.

4. **La variable `duration` debe descartarse** del modelo predictivo por data leakage, tal como recomienda la documentación original del dataset.

5. **Los valores `unknown` fueron imputados exitosamente con KNN**, preservando la estructura natural de los datos mejor que la imputación por moda.

### Recomendaciones

- **Para el modelo predictivo:** Utilizar las variables socioeconómicas como features principales, junto con `poutcome` y `contact`.
- **Para las campañas futuras:** Concentrar esfuerzos en meses con mayor tasa de conversión y en clientes con historial de éxito previo.
- **Para el escalamiento:** Aplicar StandardScaler si se utiliza regresión logística o SVM; MinMaxScaler si se opta por redes neuronales.

---
## 12. Referencias

1. Moro, S., Cortez, P., & Rita, P. (2014). *A data-driven approach to predict the success of bank telemarketing*. Decision Support Systems, 62, 22-31. doi: [10.1016/j.dss.2014.03.001](https://doi.org/10.1016/j.dss.2014.03.001)

2. UCI Machine Learning Repository. *Bank Marketing Data Set*. Disponible en: [http://archive.ics.uci.edu/ml/datasets/Bank+Marketing](http://archive.ics.uci.edu/ml/datasets/Bank+Marketing)

3. Banco de Portugal. *Estadísticas Económicas*. Disponible en: [https://www.bportugal.pt/estatisticasweb](https://www.bportugal.pt/estatisticasweb)

4. Matplotlib Documentation. Disponible en: [https://matplotlib.org/stable/](https://matplotlib.org/stable/)

5. Seaborn Documentation. Disponible en: [https://seaborn.pydata.org/](https://seaborn.pydata.org/)

6. Scikit-learn Documentation - KNNImputer. Disponible en: [https://scikit-learn.org/stable/modules/generated/sklearn.impute.KNNImputer.html](https://scikit-learn.org/stable/modules/generated/sklearn.impute.KNNImputer.html)

7. Scikit-learn Documentation - StandardScaler & MinMaxScaler. Disponible en: [https://scikit-learn.org/stable/modules/preprocessing.html](https://scikit-learn.org/stable/modules/preprocessing.html)